In [46]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_classic.retrievers import EnsembleRetriever
from sentence_transformers import CrossEncoder
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
with open("scifact/qrels/test.tsv") as f:
    all_lines=f.read().splitlines()

for i in range(5):
    print(all_lines[i])

print(len(all_lines))

query-id	corpus-id	score
1	31715818	1
3	14717500	1
5	13734012	1
13	1606628	1
340


In [4]:
qrels={}
for line in all_lines:
    line=line.split("\t")
    q_id,c_id,score = line
    if q_id not in qrels:
        qrels[q_id]=[]
    qrels[q_id].append(c_id)
    


In [5]:
qrels

{'query-id': ['corpus-id'],
 '1': ['31715818'],
 '3': ['14717500'],
 '5': ['13734012'],
 '13': ['1606628'],
 '36': ['5152028', '11705328'],
 '42': ['18174210'],
 '48': ['13734012'],
 '49': ['5953485'],
 '50': ['12580014'],
 '51': ['45638119'],
 '53': ['45638119'],
 '54': ['49556906'],
 '56': ['4709641'],
 '57': ['4709641'],
 '70': ['5956380', '4414547'],
 '72': ['6076903'],
 '75': ['4387784'],
 '94': ['1215116'],
 '99': ['18810195'],
 '100': ['4381486'],
 '113': ['6157837'],
 '115': ['33872649'],
 '118': ['6372244'],
 '124': ['4883040'],
 '127': ['21598000'],
 '128': ['8290953'],
 '129': ['27768226'],
 '130': ['27768226'],
 '132': ['7975937'],
 '133': ['38485364', '6969753', '17934082', '16280642', '12640810'],
 '137': ['26016929'],
 '141': ['6955746', '14437255'],
 '142': ['10582939'],
 '143': ['10582939'],
 '146': ['10582939'],
 '148': ['1084345'],
 '163': ['18872233'],
 '171': ['12670680'],
 '179': ['16322674', '27123743', '23557241', '17450673'],
 '180': ['16966326'],
 '183': ['128

In [6]:
corpus=[]

In [7]:
import json
with open("scifact/corpus.jsonl","r",encoding="utf-8") as f:
    for line in f:
        corpus.append(json.loads(line))

In [8]:
print(len(corpus))

5183


In [9]:
print(corpus[0])

{'_id': '4983', 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the internal capsule, the

In [10]:
print(corpus[0]["_id"])

4983


In [11]:
print(corpus[0]["metadata"])

{}


In [12]:
for entry in corpus:
    if "corpus_id" not in entry["metadata"]:
        entry["metadata"]["corpus_id"]={}
    entry["metadata"]["corpus_id"]=entry["_id"]

In [13]:
print(corpus[0])

{'_id': '4983', 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the internal capsule, the

In [14]:
docs=[]
for doc in corpus:
    docs.append(
        Document(
            page_content=f'{doc["title"]}. {doc["text"]}',
            metadata={"corpus_id": doc["_id"]}
        )
    )

In [15]:
words=len(docs[0].page_content.split())
print(words)

295


In [16]:
embeddings=OllamaEmbeddings(model="mxbai-embed-large")


In [17]:
print(len(docs))

5183


In [18]:
print(len(docs[0].page_content))
print(len(docs[100].page_content))

1942
760


In [19]:
DDB = "./Chroma-mxbai"
BATCH = 32

Dense_vs = Chroma(persist_directory=DDB, embedding_function=embeddings)
batch=1
if Dense_vs._collection.count() == len(docs):
    print("Loaded existing DB")
else:
    Dense_vs.delete_collection()
    for i in range(0, len(docs), BATCH):
        Dense_vs = Chroma.from_documents(
            docs[i:i + BATCH], embeddings, persist_directory=DDB
        )
        batch=batch+1
        print(f"batch {batch} embedded successfuly")
    print("Built DB:", Dense_vs._collection.count())


Loaded existing DB


In [20]:
"""
Dense_vs = Chroma(persist_directory=DDB, embedding_function=embeddings)

# corpus_ids already stored, read from the metadata
stored = Dense_vs.get(include=["metadatas"])["metadatas"]
done = {m["corpus_id"] for m in stored}

todo = [d for d in docs if d.metadata["corpus_id"] not in done]
print(f"{len(done)} stored, {len(todo)} remaining")

for i in range(0, len(todo), BATCH):
    Dense_vs.add_documents(todo[i:i + BATCH])
    print(f"{min(i + BATCH, len(todo))}/{len(todo)} embedded")

print("Total in DB:", Dense_vs._collection.count())   # expect 5183
"""


'\nDense_vs = Chroma(persist_directory=DDB, embedding_function=embeddings)\n\n# corpus_ids already stored, read from the metadata\nstored = Dense_vs.get(include=["metadatas"])["metadatas"]\ndone = {m["corpus_id"] for m in stored}\n\ntodo = [d for d in docs if d.metadata["corpus_id"] not in done]\nprint(f"{len(done)} stored, {len(todo)} remaining")\n\nfor i in range(0, len(todo), BATCH):\n    Dense_vs.add_documents(todo[i:i + BATCH])\n    print(f"{min(i + BATCH, len(todo))}/{len(todo)} embedded")\n\nprint("Total in DB:", Dense_vs._collection.count())   # expect 5183\n'

In [21]:
"""
Dense_vs = Chroma(persist_directory=DDB, embedding_function=embeddings)

# corpus_ids already stored, read from the metadata
stored = Dense_vs.get(include=["metadatas"])["metadatas"]
done = {m["corpus_id"] for m in stored}

todo = [d for d in docs if d.metadata["corpus_id"] not in done]
print(f"{len(done)} stored, {len(todo)} remaining")

for i in range(0, len(todo), BATCH):
    Dense_vs.add_documents(todo[i:i + BATCH])
    print(f"{min(i + BATCH, len(todo))}/{len(todo)} embedded")

print("Total in DB:", Dense_vs._collection.count())   # expect 5183
"""

'\nDense_vs = Chroma(persist_directory=DDB, embedding_function=embeddings)\n\n# corpus_ids already stored, read from the metadata\nstored = Dense_vs.get(include=["metadatas"])["metadatas"]\ndone = {m["corpus_id"] for m in stored}\n\ntodo = [d for d in docs if d.metadata["corpus_id"] not in done]\nprint(f"{len(done)} stored, {len(todo)} remaining")\n\nfor i in range(0, len(todo), BATCH):\n    Dense_vs.add_documents(todo[i:i + BATCH])\n    print(f"{min(i + BATCH, len(todo))}/{len(todo)} embedded")\n\nprint("Total in DB:", Dense_vs._collection.count())   # expect 5183\n'

In [22]:
retriever=Dense_vs.as_retriever()

In [23]:
query=[]
with open("scifact/queries.jsonl","r",encoding="utf-8") as f:
    for line in f:
        query.append(json.loads(line))

In [24]:
print(query[4])

{'_id': '9', 'text': '32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.', 'metadata': {'44265107': [{'sentences': [15], 'label': 'SUPPORT'}]}}


In [25]:
query_message=query[0]["text"]

In [26]:
response=retriever.invoke(query_message)
print(response)

[Document(id='85510d24-9794-475a-b28d-dc4b189bf9da', metadata={'corpus_id': '4346436'}, page_content='Nonlinear Elasticity in Biological Gels. Unlike most synthetic materials, biological materials often stiffen as they are deformed. This nonlinear elastic response, critical for the physiological function of some tissues, has been documented since at least the 19th century, but the molecular structure and the design principles responsible for it are unknown. Current models for this response require geometrically complex ordered structures unique to each material. In this Article we show that a much simpler molecular theory accounts for strain stiffening in a wide range of molecularly distinct biopolymer gels formed from purified cytoskeletal and extracellular proteins. This theory shows that systems of semi-flexible chains such as filamentous proteins arranged in an open crosslinked meshwork invariably stiffen at low strains without the need for a specific architecture or multiple eleme

In [27]:
recieved_doc=response[0]
print(recieved_doc.metadata["corpus_id"])

4346436


In [28]:
def is_present(corpus_id_list,response):
    found=0
    for id in corpus_id_list:
        for i in range(0,len(response)):
            recieved_doc_id=response[i].metadata["corpus_id"]
            if id==recieved_doc_id:
                found=found+1
                break
    return found/len(corpus_id_list)



In [29]:
def Recall(K):
    retriever=Dense_vs.as_retriever(search_kwargs={"k": K})
    total_recall=0
    total=0
    for q in query:
        if q["_id"] not in qrels:
            continue
        corpus_Id_list=qrels[q["_id"]]
        response=retriever.invoke(q["text"])
        total=total+1
        total_recall=total_recall+is_present(corpus_Id_list,response)
    return total_recall/total
"""
for k in [2,3,4,5,6]:
    recall=Recall(k)
    print("Recall@",k,": ",recall)
"""

'\nfor k in [2,3,4,5,6]:\n    recall=Recall(k)\n    print("Recall@",k,": ",recall)\n'

In [30]:
print(len(response))

4


In [31]:
print(response)

[Document(id='85510d24-9794-475a-b28d-dc4b189bf9da', metadata={'corpus_id': '4346436'}, page_content='Nonlinear Elasticity in Biological Gels. Unlike most synthetic materials, biological materials often stiffen as they are deformed. This nonlinear elastic response, critical for the physiological function of some tissues, has been documented since at least the 19th century, but the molecular structure and the design principles responsible for it are unknown. Current models for this response require geometrically complex ordered structures unique to each material. In this Article we show that a much simpler molecular theory accounts for strain stiffening in a wide range of molecularly distinct biopolymer gels formed from purified cytoskeletal and extracellular proteins. This theory shows that systems of semi-flexible chains such as filamentous proteins arranged in an open crosslinked meshwork invariably stiffen at low strains without the need for a specific architecture or multiple eleme

In [32]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever1 = BM25Retriever.from_documents(docs, k=6)


C:\Users\ayusagga\AppData\Local\Temp\ipykernel_28636\1041896004.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


In [33]:
def tokenize(text):
    text=text.lower()
    for ch in ".,/;:\"|%-()[]'?":
        text=text.replace(ch," ")
    return text.split()

In [34]:
bm25_retriever2 = BM25Retriever.from_documents(docs, preprocess_func=tokenize, k=6)


In [35]:
bm25_retriever2.invoke(query[0]["text"])

[Document(metadata={'corpus_id': '40212412'}, page_content="Periosteal bone formation--a neglected determinant of bone strength.. Life forms that have low body mass can hunt for food on the undersurface of branches or along shear cliff faces quite unperturbed by gravity. For larger animals, the hunt for dinner and the struggle to avoid becoming someone else's meal require rapid movement against gravity. This need is met by the lever function of long bones, three-dimensional masterpieces of biomechanical engineering that, by their material composition and structural design, achieve the contradictory properties of stiffness and flexibility, strength and lightness.1 Material stiffness results from the encrusting of the triple-helical structure of collagen type I with hydroxyapatite crystals, which confers . . ."),
 Document(metadata={'corpus_id': '825728'}, page_content='Metastatic colonization requires the repression of the epithelial-mesenchymal transition inducer Prrx1.. The epithelial

In [36]:
def Recall_on_BM25(retriever):
    total=0
    total_recall=0
    for q in query:
        if q["_id"] not in qrels:
            continue
        corpus_Id_list=qrels[q["_id"]]
        response=retriever.invoke(q["text"])
        total=total+1
        total_recall=total_recall+is_present(corpus_Id_list,response)

    return total_recall/total



In [37]:
"""for kval in [2,3,4,5,6,7,8,9,10]:
    bm25_retriever=BM25Retriever.from_documents(docs,preprocess_func=tokenize,k=kval)
    recall=Recall_on_BM25(bm25_retriever)
    print("BM25-->Recall@",kval,": ",recall)"""

'for kval in [2,3,4,5,6,7,8,9,10]:\n    bm25_retriever=BM25Retriever.from_documents(docs,preprocess_func=tokenize,k=kval)\n    recall=Recall_on_BM25(bm25_retriever)\n    print("BM25-->Recall@",kval,": ",recall)'

### Combining the dense and sparse vectors

In [38]:
bm25_pool = BM25Retriever.from_documents(docs, preprocess_func=tokenize, k=30)
dense_pool = Dense_vs.as_retriever(search_kwargs={"k": 30})

hybrid = EnsembleRetriever(retrievers=[bm25_pool, dense_pool], weights=[0.5, 0.5])

response = hybrid.invoke(query[0]["text"])[:6]


In [39]:
print(query[0])

{'_id': '0', 'text': '0-dimensional biomaterials lack inductive properties.', 'metadata': {}}


In [40]:
hybrid_response={}
for q in query:
    if q["_id"] not in qrels:
        continue
    hybrid_response[q["_id"]]=hybrid.invoke(q["text"])


KeyboardInterrupt: 

In [40]:
def Hybrid_Recall(kval):   
    total=0
    total_recall=0
    for q in hybrid_response:
        corpus_list=qrels[q]
        response=hybrid_response[q][:kval]
        total=total+1
        total_recall=total_recall+is_present(corpus_list,response)
    return total_recall/total


In [52]:
for kval in [2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20]:
    recall=Hybrid_Recall(kval)
    print("Hybrid_recall@",kval,": ",recall)

Hybrid_recall@ 2 :  0.6970555555555555
Hybrid_recall@ 3 :  0.7432777777777777
Hybrid_recall@ 4 :  0.7710555555555555
Hybrid_recall@ 5 :  0.7893888888888889
Hybrid_recall@ 6 :  0.8164444444444444
Hybrid_recall@ 7 :  0.8314444444444443
Hybrid_recall@ 8 :  0.843111111111111
Hybrid_recall@ 9 :  0.8619444444444444
Hybrid_recall@ 10 :  0.863611111111111
Hybrid_recall@ 11 :  0.8709444444444443
Hybrid_recall@ 12 :  0.885111111111111
Hybrid_recall@ 13 :  0.8885555555555555
Hybrid_recall@ 14 :  0.8885555555555555
Hybrid_recall@ 15 :  0.8977222222222222
Hybrid_recall@ 16 :  0.9010555555555555
Hybrid_recall@ 17 :  0.9010555555555555
Hybrid_recall@ 19 :  0.9143888888888889
Hybrid_recall@ 20 :  0.9167222222222222


### Using Reranker for better results


In [55]:
reranker=CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

c:\GenAI-course\Hybrid_search\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ayusagga\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\GenAI-course\Hybrid_search\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: U

In [57]:
qid=list(hybrid_response)[0]

In [62]:
POOL=20

query_text={}
for q in query:
    query_text[q["_id"]]=q["text"]

def get_score(item):
    return item[0]
reranked_response={}

for qid in hybrid_response:
    candidates=hybrid_response[qid][:POOL]
    pairs=[]
    for doc in candidates:
        pairs.append((query_text[qid],doc.page_content))
    score=reranker.predict(pairs)
    scored=[]
    for i in range(len(candidates)):
        scored.append((score[i],candidates[i]))

    scored.sort(key=get_score,reverse=True)

    new_order=[]
    for s,doc in scored:
        new_order.append(doc)
    reranked_response[qid]=new_order

In [60]:
reranked_response[qid]

[Document(metadata={'corpus_id': '43385013'}, page_content='Epithelial and mesenchymal subpopulations within normal basal breast cell lines exhibit distinct stem cell/progenitor properties.. It has been proposed that epithelial-mesenchymal transition (EMT) in mammary epithelial cells and breast cancer cells generates stem cell features, and that the presence of EMT characteristics in claudin-low breast tumors reveals their origin in basal stem cells. It remains to be determined, however, whether EMT is an inherent property of normal basal stem cells, and if the presence of a mesenchymal-like phenotype is required for the maintenance of all their stem cell properties. We used nontumorigenic basal cell lines as models of normal stem cells/progenitors and demonstrate that these cell lines contain an epithelial subpopulation ("EpCAM+," epithelial cell adhesion molecule positive [EpCAM(pos)]/CD49f(high)) that spontaneously generates mesenchymal-like cells ("Fibros," EpCAM(neg)/CD49f(med/low

In [63]:

def reranker_Recall(kval):
    total=0
    total_recall=0
    for qid in reranked_response:
        corpus_list=qrels[qid]
        response=reranked_response[qid][:kval]
        total=total+1
        total_recall=total_recall + is_present(corpus_list,response)
    return total_recall/total

In [64]:
for k in [2,3,4,5,6,7,8,9,10,11,12]:
    recall=reranker_Recall(k)
    print("Reranker_Recall@",k,": ",recall)

Reranker_Recall@ 2 :  0.6506666666666666
Reranker_Recall@ 3 :  0.699111111111111
Reranker_Recall@ 4 :  0.7408888888888888
Reranker_Recall@ 5 :  0.7701666666666666
Reranker_Recall@ 6 :  0.7891666666666666
Reranker_Recall@ 7 :  0.8052777777777778
Reranker_Recall@ 8 :  0.8226111111111111
Reranker_Recall@ 9 :  0.8409444444444444
Reranker_Recall@ 10 :  0.8559444444444444
Reranker_Recall@ 11 :  0.8609444444444443
Reranker_Recall@ 12 :  0.8710555555555556


In [66]:
recall=reranker_Recall(20)
print("Reranker_Recall@",20,": ",recall)

Reranker_Recall@ 20 :  0.9167222222222222


### Trying different Reranker

In [50]:
reranker = CrossEncoder("BAAI/bge-reranker-base")


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 512.22it/s]


In [68]:
POOL=10

query_text={}
for q in query:
    query_text[q["_id"]]=q["text"]

def get_score(item):
    return item[0]
reranked_response={}

for qid in hybrid_response:
    candidates=hybrid_response[qid][:POOL]
    pairs=[]
    for doc in candidates:
        pairs.append((query_text[qid],doc.page_content))
    score=reranker.predict(pairs)
    scored=[]
    for i in range(len(candidates)):
        scored.append((score[i],candidates[i]))

    scored.sort(key=get_score,reverse=True)

    new_order=[]
    for s,doc in scored:
        new_order.append(doc)
    reranked_response[qid]=new_order

In [69]:

def reranker_Recall(kval):
    total=0
    total_recall=0
    for qid in reranked_response:
        corpus_list=qrels[qid]
        response=reranked_response[qid][:kval]
        total=total+1
        total_recall=total_recall + is_present(corpus_list,response)
    return total_recall/total

In [70]:
for k in [2,3,4,5,6,7,8,9,10]:
    recall=reranker_Recall(k)
    print("Reranker_Recall@",k,": ",recall)

Reranker_Recall@ 2 :  0.6913333333333334
Reranker_Recall@ 3 :  0.7495
Reranker_Recall@ 4 :  0.7982222222222222
Reranker_Recall@ 5 :  0.8173888888888888
Reranker_Recall@ 6 :  0.8374999999999999
Reranker_Recall@ 7 :  0.851111111111111
Reranker_Recall@ 8 :  0.8552777777777777
Reranker_Recall@ 9 :  0.8586111111111111
Reranker_Recall@ 10 :  0.863611111111111


### Best response is seems to be Hybrid(RRF) + bge-reranker-base

In [41]:
response = hybrid.invoke(query[0]["text"])[:10]

In [42]:
print(response)

[Document(metadata={'corpus_id': '10607877'}, page_content='Mechanical modulation of receptor-ligand interactions at cell-cell interfaces.. Cell surface receptors have been extensively studied because they initiate and regulate signal transduction cascades leading to a variety of functional cellular outcomes. An important class of immune receptors (e.g., T-cell antigen receptors) whose ligands are anchored to the surfaces of other cells remain poorly understood. The mechanism by which ligand binding initiates receptor phosphorylation, a process termed "receptor triggering", remains controversial. Recently, direct measurements of the (two-dimensional) receptor-ligand complex lifetimes at cell-cell interface were found to be smaller than (three-dimensional) lifetimes in solution but the underlying mechanism is unknown. At the cell-cell interface, the receptor-ligand complex spans a short intermembrane distance (15 nm) compared to long surface molecules (LSMs) whose ectodomains span >40 n

In [66]:
qa_message=""" 
You are a scientific fact-checking assistant.
Using the below context, answer the user's claim is VALID/INVALID with the 
with few lines of information about it. Also give the citation ID.
For the claims outside of the given context, simply say NOT ENOUGH INFO PRESENT.
Do not use any knowledge outside the context.

context: {context}

"""

qa_prompt=ChatPromptTemplate.from_messages([
    ('system',qa_message),
    ('human', '{input}')
])

In [67]:
llm=ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [68]:
def get_score(item):
    return item[0]

POOL=10
topK=6

def retrieve_docs(x):
    question=x["input"]
    doc_base=hybrid.invoke(question)[:POOL]
    pairs=[]
    for doc in doc_base:
        pairs.append((question,doc.page_content))

    scores=reranker.predict(pairs)
    scored=[]
    for i in range(0,len(doc_base)):
        scored.append((scores[i],doc_base[i]))
    scored.sort(key=get_score,reverse=True)
    reranked_list=[]
    for s,doc in scored[:topK]:
        reranked_list.append(doc)
    return reranked_list

reranked_retriever=RunnableLambda(retrieve_docs)

In [69]:
from langchain_core.prompts import PromptTemplate

doc_prompt =PromptTemplate.from_template("[{corpus_id}] {page_content}")
qa_chain=create_stuff_documents_chain(llm,qa_prompt,document_prompt=doc_prompt)
rag_chain=create_retrieval_chain(reranked_retriever,qa_chain)

In [ ]:
input="What is photosynthesis?"

In [71]:
result=rag_chain.invoke({"input":"{input}"})
print(result["answer"])

NOT ENOUGH INFO PRESENT.
